# Bộ dữ liệu luận văn · vn-fill

  1 lpm-videos            244 LPM videos (from thivntrung/dataset-video), checked, train/held-out split
  2 slidespeech-devtest   SlideSpeech dev + test videos and the related files (OCR, transcripts)
  3 slidespeech-train100  100 SlideSpeech train videos, streamed out of the 232 GB archive
  4 vn-test12             12 Vietnamese lecture videos from 5 channel types + SponsorBlock labels
  5 m3av-maths            M3AV Oxford Mathematics: slide OCR/LaTeX labels + the 51 videos


In [ ]:
import csv, glob, json, os, shutil, subprocess, sys, time
from pathlib import Path

def sh(cmd, check=False):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if check and r.returncode:
        raise RuntimeError(r.stderr[-800:])
    return r

def probe(path):
    r = sh(['ffprobe', '-v', 'error', '-print_format', 'json', '-show_streams', '-show_format', str(path)])
    try:
        j = json.loads(r.stdout)
    except Exception:
        return None
    v = [s for s in j.get('streams', []) if s.get('codec_type') == 'video']
    a = [s for s in j.get('streams', []) if s.get('codec_type') == 'audio']
    return {'vcodec': v[0].get('codec_name', '') if v else '', 'width': int(v[0].get('width', 0)) if v else 0,
            'height': int(v[0].get('height', 0)) if v else 0, 'has_audio': bool(a),
            'acodec': a[0].get('codec_name', '') if a else '',
            'duration': round(float(j.get('format', {}).get('duration', 0) or 0), 2)}

def pyav_decodes(path, at=30.0):
    """True if PyAV (what stage 01 uses) can decode a frame of this file."""
    try:
        import av
        with av.open(str(path)) as c:
            s = c.streams.video[0]
            c.seek(int(at / float(s.time_base)), stream=s, backward=True, any_frame=False)
            for frame in c.decode(s):
                frame.to_ndarray(format='rgb24')
                return True
    except Exception as e:
        print('  PyAV lỗi', path.name, str(e)[:120])
    return False

def write_csv(path, rows):
    rows = list(rows)
    with open(path, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)

def du_gb(path):
    return sum(f.stat().st_size for f in Path(path).rglob('*') if f.is_file()) / 2**30

def install_ytdlp():
    sh([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'yt-dlp[default]'])
    if not shutil.which('deno'):
        os.system('curl -fsSL https://deno.land/install.sh | sh -s -- -y > /dev/null 2>&1')
        os.environ['PATH'] = os.path.expanduser('~/.deno/bin') + ':' + os.environ['PATH']
    print(sh([sys.executable, '-m', 'yt_dlp', '--version']).stdout.strip(), shutil.which('deno'))

BLOCKED_IN_A_ROW = [0]


def youtube(video_id, out, max_height=720, max_sec=None, tries=4, cookies=None, pause=(4, 10)):
    """mp4 with H.264 video (PyAV/OpenCV read it everywhere) and AAC audio; cut to max_sec.
    Uses deno explicitly for YouTube's JS challenge; pauses between videos; after 3 videos in a
    row are refused ("Sign in to confirm you're not a bot") it returns 'blocked' at once."""
    import random
    url = f'https://www.youtube.com/watch?v={video_id}'
    if BLOCKED_IN_A_ROW[0] >= 3:
        return {'status': 'blocked', 'reason': 'dừng: YouTube chặn 3 video liên tiếp'}
    time.sleep(random.uniform(*pause))
    deno = shutil.which('deno') or os.path.expanduser('~/.deno/bin/deno')
    js = ['--js-runtimes', f'deno:{deno}'] if os.path.exists(deno) else []
    formats = [f'bv*[height<={max_height}][vcodec^=avc1]+ba[ext=m4a]',
               f'bv*[height<={max_height}][ext=mp4]+ba[ext=m4a]', f'bv*[height<={max_height}]+ba/b']
    target = Path(out) / f'{video_id}.mp4'
    last = ''
    for attempt in range(tries):
        for f in Path(out).glob(f'{video_id}.*'):
            f.unlink()
        r = sh([sys.executable, '-m', 'yt_dlp', '-q', '--no-warnings', '--ignore-config', *js, '--no-playlist',
                '--socket-timeout', '30', '-f', formats[min(attempt, 2)],
                '--merge-output-format', 'mp4', '--retries', '10', '--fragment-retries', '10',
                *(([('--cookies-from-browser' if cookies[0] == 'browser' else '--cookies'), cookies[1]]) if cookies else []),
                '-o', str(target), url])
        if target.exists():
            info = probe(target)
            if info and info['height'] and info['has_audio']:
                if max_sec and info['duration'] > max_sec + 5:
                    cut = target.with_suffix('.cut.mp4')
                    sh(['ffmpeg', '-v', 'error', '-y', '-i', str(target), '-t', str(max_sec), '-c', 'copy', str(cut)], check=True)
                    original = info['duration']
                    cut.replace(target); info = probe(target); info['original_duration'] = original
                BLOCKED_IN_A_ROW[0] = 0
                return {'status': 'ok', **info}
            last = f'thiếu hình/tiếng: {info}'
        else:
            last = (r.stderr or '').strip().splitlines()[-1:] or ['không có file']
            last = last[0][:200]
            if 'Sign in' in last or 'not a bot' in last:
                BLOCKED_IN_A_ROW[0] += 1
                return {'status': 'blocked', 'reason': last}
        time.sleep(10 * (attempt + 1))
    return {'status': 'failed', 'reason': last}


In [ ]:
install_ytdlp()
OUT = Path('/kaggle/working/vn-fill'); (OUT / 'video').mkdir(parents=True, exist_ok=True)
MAX_SEC = 45 * 60
# 12 video, 5 kiểu: truyền hình (quảng cáo, nghỉ), báo ôn thi năm khác L25, livestream (nói ngoài lề,
# nghỉ, viết tay), bài giảng đại học có slide, lập trình. Video dài hơn 45 phút được cắt lấy 45 phút đầu.
VIDEOS = [
    ('tHE1mC_n0-E', 'truyen_hinh', 'HTV - Đài Hà Nội', 'Vật lí', 'Giải đề thi môn Vật lí · tốt nghiệp THPT 2025'),
    ('D_kuulxKuxc', 'truyen_hinh', 'VTV7', 'Toán', 'Toán 2 · Bài 12: Phép trừ (có nhớ) trong phạm vi 20'),  # thay K2qJUcj51zQ (bị gỡ)
    ('bstCZZUMJYw', 'bao_on_thi', 'Báo Thanh Niên', 'Toán', 'Bí quyết ôn thi THPT 2025 · Chuyên đề 9: Xác suất có điều kiện'),
    ('_eDGTHhyQ7M', 'bao_on_thi', 'Báo Thanh Niên', 'Hóa học', 'Bí quyết ôn thi THPT 2025 · Chuyên đề 1: Kim loại nhóm IA, IIA'),
    ('8WnDooAH2a4', 'livestream', 'Thầy Vũ Ngọc Anh', 'Vật lí', 'Chữa đề chính thức tốt nghiệp THPT 2025'),
    ('szRBlpJ0kkY', 'livestream', 'Thầy Nguyễn Quốc Chí', 'Toán', 'Xác suất có điều kiện (phần 1) · lớp 12'),
    ('laT0CKatosY', 'livestream', 'Cpen Team', 'Toán', 'ĐGNL HCM · Toán · Buổi 1'),
    ('xysJI15jpAw', 'dai_hoc', 'Quang Trung TV', 'Xác suất thống kê', 'Chương 2 phần 1: Biến ngẫu nhiên rời rạc'),
    ('v2YTz8ToEag', 'dai_hoc', 'Success and Victory (TS. Phan Thế Công)', 'Kinh tế', 'Bài 1. Tổng quan về kinh tế vi mô'),
    ('hVU5Mw19V_c', 'dai_hoc', 'Nguyễn Thành Nam', 'Xác suất thống kê', 'Công thức xác suất đầy đủ và công thức Bayes'),
    ('xLGGWMDJRkY', 'lap_trinh', 'Tri Thức Mở by The Compiler', 'Lập trình', 'CS50P bài giảng 2: Vòng lặp (tiếng Việt)'),
    ('y1CvB950VlU', 'lap_trinh', 'Gà Lại Lập Trình', 'Lập trình', 'Bài 25: String Python part 1'),
]
VIDEOS = [v for v in VIDEOS if v[0] == 'D_kuulxKuxc']
rows = []
for vid, kind, channel, subject, title in VIDEOS:
    result = youtube(vid, OUT / 'video', max_sec=MAX_SEC)
    print(vid, kind, result.get('status'), result.get('duration'), result.get('height'), result.get('reason', ''), flush=True)
    rows.append({'video_id': vid, 'kind': kind, 'channel': channel, 'subject': subject, 'title': title,
                 'youtube_url': f'https://www.youtube.com/watch?v={vid}', 'file': f'video/{vid}.mp4' if result['status'] == 'ok' else '',
                 **{k: result.get(k, '') for k in ('status', 'duration', 'original_duration', 'width', 'height', 'vcodec', 'acodec', 'reason')}})
    if result['status'] == 'blocked':
        print('YouTube chặn tải từ Kaggle; các video còn lại tải ở máy cá nhân.');
write_csv(OUT / 'videos.csv', rows)
# Ảnh xem nhanh: 8 khung rải đều mỗi video
import cv2, numpy as np
(OUT / 'contact').mkdir(exist_ok=True)
for r in rows:
    if r['status'] != 'ok':
        continue
    cap = cv2.VideoCapture(str(OUT / r['file'])); n = cap.get(cv2.CAP_PROP_FRAME_COUNT); tiles = []
    for k in range(8):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(n * (k + 0.5) / 8)); ok, f = cap.read()
        if ok:
            tiles.append(cv2.resize(f, (320, 180)))
    if tiles:
        while len(tiles) % 4: tiles.append(np.zeros_like(tiles[0]))
        cv2.imwrite(str(OUT / 'contact' / f"{r['video_id']}.jpg"), np.vstack([np.hstack(tiles[i:i + 4]) for i in range(0, len(tiles), 4)]))
from collections import Counter
summary = {'videos_ok': sum(r['status'] == 'ok' for r in rows), 'videos': len(rows),
           'hours': round(sum(float(r['duration'] or 0) for r in rows) / 3600, 2),
           'by_kind': dict(Counter(r['kind'] for r in rows if r['status'] == 'ok')), 'size_gb': round(du_gb(OUT), 2)}
json.dump(summary, open(OUT / 'summary.json', 'w'), ensure_ascii=False, indent=1)
print(summary)
